In [19]:
import torch
import torchvision
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms
from collections import Counter
from torch.utils.data import random_split, DataLoader
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset

# Task 1. Study and Prepare the Data

In [ ]:
# Download and load CIFAR-10
train_dataset = torchvision.datasets.CIFAR10(root="./data", train=True, download=True,transform=transforms.ToTensor())

test_dataset = torchvision.datasets.CIFAR10(root="./data", train=False, download=True,transform=transforms.ToTensor())

In [ ]:
# Analyze the dataset
classes = train_dataset.classes

print("Training samples:", len(train_dataset))
print("Test samples:", len(test_dataset))
print("Image dimensions:", train_dataset.data.shape[1:])
print("Number of classes:", len(classes))
print("Classes:", classes)

In [ ]:
# Inspect class distribution
class_counts = Counter(train_dataset.targets)

for i, class_name in enumerate(classes):
    print(f"{class_name}: {class_counts[i]}")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))

for i, ax in enumerate(axes.flat):
    image = train_dataset.data[i]
    label = train_dataset.targets[i]

    ax.imshow(image)
    ax.set_title(classes[label])
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Normalization
# Source: https://stackoverflow.com/questions/66678052/how-to-calculate-the-mean-and-the-std-of-cifar10-data
mean = [0.49139968, 0.48215827 ,0.44653124]
std =  [0.24703233, 0.24348505, 0.26158768]
train_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean, std)])

eval_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean, std)])

In [15]:
# Create all 3 splits
seed = 587
rng = np.random.default_rng(seed)

train_subset, val_subset = random_split(
    train_data, [40000, 10000],
    generator=torch.Generator().manual_seed(seed)
)
 # Test already defined in the first cell

print("Training:", len(train_subset))
print("Validation:", len(val_subset))
print("Test:", len(test_data))

Training: 40000
Validation: 10000
Test: 10000


In [18]:
# Data loaders for efficient batch processing
train_loader = DataLoader(train_subset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=128, shuffle=False)
test_loader = DataLoader(test_data, batch_size=128, shuffle=False)

# Task 2. Implement and Compare the Models


In [23]:
if torch.cuda.is_available(): device = torch.device("cuda")
elif torch.backends.mps.is_available(): device = torch.device("mps")
else: device = torch.device("cpu")

EPOCHS = 20
LEARNING_RATE = 0.001

In [24]:
# AlexNet
class AlexNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.features = nn.Sequential(
            # Conv1
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            # Conv2
            nn.Conv2d(64, 192, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            # Conv3
            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            # Conv4
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            # Conv5
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),

            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),

            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x